In [35]:
import numpy as np
import pandas as pd
import torch
from torch import nn

data = pd.read_csv('train.csv')

In [36]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
data = data.to_numpy()
# 0th col are the labels of the images
# 1st col up to 784th col are pixel values

In [38]:
data

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [7, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [9, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [39]:
labels = data[:, 0]
data = data[:, 1:785]

In [40]:
data.shape

(42000, 784)

In [41]:
train_split = int(0.8 * len(data))
y_train, y_test = labels[:train_split], labels[train_split:]
X_train, X_test = data[:train_split], data[train_split:]

In [42]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train, dtype=torch.int64), torch.tensor(y_test, dtype=torch.int64)
X_train, X_test = X_train/255, X_test/255

In [43]:
class NumClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 10)
        self.layer2 = nn.Linear(10, 10)
        self.layer3 = nn.Linear(10,10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

In [44]:
model_1 = NumClassificationModel()

In [45]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [46]:
X_train.shape

torch.Size([33600, 784])

In [47]:
y_logits = model_1(X_train)
print(y_logits, y_logits.shape)
y_pred = nn.Softmax(dim=1)(y_logits)
print(y_pred, y_pred.shape)
y_pred_labels = torch.argmax(y_pred, dim=1)
print(y_pred_labels, y_pred_labels.shape)

tensor([[-0.2592,  0.0232,  0.2449,  ...,  0.2437,  0.1363, -0.1841],
        [-0.2586,  0.0177,  0.2431,  ...,  0.2459,  0.1282, -0.1888],
        [-0.2652,  0.0130,  0.2448,  ...,  0.2495,  0.1319, -0.1904],
        ...,
        [-0.2489,  0.0216,  0.2455,  ...,  0.2469,  0.1205, -0.1905],
        [-0.2703,  0.0164,  0.2481,  ...,  0.2493,  0.1439, -0.1857],
        [-0.2512,  0.0303,  0.2432,  ...,  0.2387,  0.1338, -0.1811]],
       grad_fn=<AddmmBackward0>) torch.Size([33600, 10])
tensor([[0.0783, 0.1038, 0.1296,  ..., 0.1295, 0.1163, 0.0844],
        [0.0785, 0.1035, 0.1297,  ..., 0.1301, 0.1156, 0.0842],
        [0.0781, 0.1031, 0.1301,  ..., 0.1307, 0.1162, 0.0842],
        ...,
        [0.0792, 0.1038, 0.1299,  ..., 0.1301, 0.1146, 0.0840],
        [0.0775, 0.1033, 0.1302,  ..., 0.1303, 0.1173, 0.0844],
        [0.0788, 0.1044, 0.1292,  ..., 0.1286, 0.1158, 0.0845]],
       grad_fn=<SoftmaxBackward0>) torch.Size([33600, 10])
tensor([2, 7, 7,  ..., 7, 7, 2]) torch.Size([33600])

In [48]:
def accuracy(actual_labels, pred_labels):
    '''
    both actual_labels and pred_labels are vectors where each entry is an example
    '''
    return (pred_labels == actual_labels).sum().item() / len(actual_labels) * 100

In [49]:
# training loop
epochs = 1000 

for epoch in range(epochs):
    model_1.train()
    
    # forward pass
    y_logits = model_1(X_train)
    y_pred = nn.Softmax(dim=1)(y_logits)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    # calculate the loss
    loss = loss_fn(y_logits, y_train)
    acc = accuracy(y_train, y_pred_labels)

    # optmizier.zero_grad()
    optimizer.zero_grad()

    # loss.backward()
    loss.backward()

    # optimizer.step()
    optimizer.step()

    # testing loop
    model_1.eval()
    with torch.inference_mode():
        # forward pass
        y_test_logits = model_1(X_test)
        y_test_pred = nn.Softmax(dim=1)(y_test_logits)
        y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

        # calculate the loss
        test_loss = loss_fn(y_test_logits, y_test)
        test_acc = accuracy(y_test, y_test_pred_labels)

        if epoch % 20 == 0: 
            print(f'epoch: {epoch} | train loss: {loss:.5f} | train acc: {acc}% | test loss: {test_loss:.5f} | test acc: {test_acc}%')


epoch: 0 | train loss: 2.31792 | train acc: 13.247023809523808% | test loss: 2.31833 | test acc: 13.440476190476192%
epoch: 20 | train loss: 2.29655 | train acc: 17.479166666666668% | test loss: 2.29622 | test acc: 17.36904761904762%
epoch: 40 | train loss: 2.26746 | train acc: 19.6875% | test loss: 2.26566 | test acc: 19.78571428571429%
epoch: 60 | train loss: 2.20681 | train acc: 22.639880952380953% | test loss: 2.20126 | test acc: 23.761904761904763%
epoch: 80 | train loss: 2.07635 | train acc: 28.31845238095238% | test loss: 2.06341 | test acc: 28.76190476190476%
epoch: 100 | train loss: 1.85479 | train acc: 36.01488095238095% | test loss: 1.83219 | test acc: 37.11904761904762%
epoch: 120 | train loss: 1.58696 | train acc: 51.2470238095238% | test loss: 1.56054 | test acc: 52.78571428571428%
epoch: 140 | train loss: 1.33138 | train acc: 61.2529761904762% | test loss: 1.30775 | test acc: 62.095238095238095%
epoch: 160 | train loss: 1.11153 | train acc: 67.93154761904762% | test loss